<a href="https://colab.research.google.com/github/greek-nlp/benchmark/blob/main/suite_benchmark_colab_monte_carlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Greek NLP Benchmark Suite in Colab

This notebook is the Colab entrypoint for the benchmark tasks in this repository.

Current state:
- The notebook sets up Ollama inside Colab.
- It organizes the full task suite in one place.
- The repository now includes a shared runner in `suite_benchmark.py`.
- Direct shared-runner tasks currently include GEC, toxicity, machine translation, intent classification, and summarization.
- The remaining tasks still point to the task-specific notebooks already present in the repo.


## Quick Start

Run the notebook top to bottom:
1. Setup cells
2. Suite configuration
3. Model pull cell
4. Task overview
5. GEC benchmark cells

Important: in Ollama, use `llama3.1:8b`, not `llama3.1:8b-instruct`.


## Repo Setup

Open this notebook from the repository in Colab, or clone/upload the repository files into the runtime before continuing.


## 1. Environment Setup

In [1]:
!apt-get -qq update
!apt-get -qq install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!pip -q install pandas pywer openpyxl datasets scikit-learn

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print("Started Ollama server with PID", ollama_process.pid)

Started Ollama server with PID 12912


## 2. Suite Configuration

In [5]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
print("Working directory:", project_root)
print("Python executable:", sys.executable)

MODEL_PRESETS = {
    "t4": [
        "qwen2.5:3b",
        "gemma2:2b",
        "mistral:7b",
        "qwen2.5:7b-instruct",
    ],
    "high_memory": [
        "llama3.1:8b",
        "aya-expanse:8b",
        "gemma2:9b",
        "qwen2.5:14b",
    ],
}

TASKS = {
    "toxicity": {
        "datasets": ["OGTD / OffenseEval Greek"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "gec": {
        "datasets": ["GNC / Korre"],
        "entrypoints": ["suite_benchmark_colab_monte_carlo.ipynb", "gec_benchmark_notebook.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "machine_translation": {
        "datasets": ["PGV"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "summarization": {
        "datasets": ["GreekLegalSum"],
        "entrypoints": ["nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "intent_classification": {
        "datasets": ["UniWay GR"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "ner": {
        "datasets": ["elNER", "ner_nel_greek_dataset", "UniWay GR"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_access_data.ipynb"],
        "runner": "notebook",
    },
    "pos_tagging": {
        "datasets": ["UD Greek GDT"],
        "entrypoints": ["nlp_gr_access_data.ipynb"],
        "runner": "notebook",
    },
    "clustering": {
        "datasets": ["Greek Legal Code"],
        "entrypoints": ["nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
    "authorship_attribution": {
        "datasets": ["author datasets in repo notebooks"],
        "entrypoints": ["gr_attribution_zeroshot.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
}

MODEL_PRESET = "t4"
MODELS = MODEL_PRESETS[MODEL_PRESET]
ACTIVE_TASKS = list(TASKS)

SAMPLE_SIZE = 100
NUM_REPEATS = 5
RANDOM_STATE = 42
OUTPUT_ROOT = Path("results/colab_suite")

print("Model preset:", MODEL_PRESET)
print("Models:", MODELS)
print("Tasks:", ACTIVE_TASKS)
print("Sample size:", SAMPLE_SIZE)
print("Repeats per task:", NUM_REPEATS)


Working directory: /content
Python executable: /usr/bin/python3
Model preset: t4
Models: ['qwen2.5:3b', 'gemma2:2b', 'mistral:7b', 'qwen2.5:7b-instruct']
Tasks: ['toxicity', 'gec', 'machine_translation', 'summarization', 'intent_classification', 'ner', 'pos_tagging', 'clustering', 'authorship_attribution']


In [6]:
for model in MODELS:
    print(f"Pulling {model}...")
    subprocess.run(["ollama", "pull", model], check=True)

Pulling qwen2.5:3b...
Pulling gemma2:2b...
Pulling mistral:7b...
Pulling qwen2.5:7b-instruct...


In [7]:
!ollama list

NAME                   ID              SIZE      MODIFIED               
qwen2.5:7b-instruct    845dbda0ea48    4.7 GB    Less than a second ago    
mistral:7b             6577803aa9a0    4.4 GB    27 seconds ago            
gemma2:2b              8ccf136fdd52    1.6 GB    57 seconds ago            
qwen2.5:3b             357c53fb659c    1.9 GB    About a minute ago        


## 3. Task Overview

In [8]:
task_rows = []
for task_name, info in TASKS.items():
    task_rows.append(
        {
            "task": task_name,
            "runner": info["runner"],
            "datasets": ", ".join(info["datasets"]),
            "entrypoints": ", ".join(info["entrypoints"]),
        }
    )

pd.DataFrame(task_rows)

,task,runner,datasets,entrypoints
0,toxicity,shared_runner,OGTD / OffenseEval Greek,"air_gr_nlp_benchmark.ipynb, nlp_gr_experiments..."
1,gec,shared_runner,GNC / Korre,"suite_benchmark_colab.ipynb, gec_benchmark_not..."
2,machine_translation,shared_runner,PGV,"air_gr_nlp_benchmark.ipynb, nlp_gr_experiments..."
3,summarization,shared_runner,GreekLegalSum,nlp_gr_experiments.ipynb
4,intent_classification,shared_runner,UniWay GR,"air_gr_nlp_benchmark.ipynb, nlp_gr_experiments..."
5,ner,notebook,"elNER, ner_nel_greek_dataset, UniWay GR","air_gr_nlp_benchmark.ipynb, nlp_gr_access_data..."
6,pos_tagging,notebook,UD Greek GDT,nlp_gr_access_data.ipynb
7,clustering,notebook,Greek Legal Code,nlp_gr_experiments.ipynb
8,authorship_attribution,notebook,author datasets in repo notebooks,"gr_attribution_zeroshot.ipynb, nlp_gr_experime..."


## 4. Notebook Routing for Non-Shared Tasks

The shared runner currently covers GEC, toxicity, machine translation, intent classification, and summarization.

For the remaining tasks, start from these notebooks:
- NER: `air_gr_nlp_benchmark.ipynb`, `nlp_gr_access_data.ipynb`
- POS tagging: `nlp_gr_access_data.ipynb`
- Clustering: `nlp_gr_experiments.ipynb`
- Authorship attribution: `gr_attribution_zeroshot.ipynb`, `nlp_gr_experiments.ipynb`


## 5. Run the Shared Benchmark Suite and Inspect Results Per Task

In [9]:
import os

repo_url = "https://github.com/greek-nlp/benchmark.git"
repo_dir = "benchmark"

# Clone the repository if it doesn't already exist
if not os.path.exists(repo_dir):
    print(f"Cloning {repo_url} into {repo_dir}...")
    !git clone {repo_url}
    # Change current working directory to the cloned repository
    %cd {repo_dir}
    print(f"Changed current working directory to {os.getcwd()}")
else:
    print(f"Repository {repo_dir} already exists. Pulling latest changes...")
    # Change into the directory and pull latest changes
    %cd {repo_dir}
    !git pull
    print(f"Current working directory is {os.getcwd()}")

Cloning https://github.com/greek-nlp/benchmark.git into benchmark...
Cloning into 'benchmark'...
remote: Enumerating objects: 311, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 311 (delta 95), reused 36 (delta 15), pack-reused 146 (from 1)
Receiving objects: 100% (311/311), 20.40 MiB | 17.97 MiB/s, done.
Resolving deltas: 100% (174/174), done.
/content/benchmark
Changed current working directory to /content/benchmark


In [10]:
import sys

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from benchmark_suite import GenerationConfig, list_tasks, run_task, save_run_outputs

def aggregate_repeated_summaries(summary_by_repeat):
    repeated_summary = pd.concat(summary_by_repeat, ignore_index=True)
    key_columns = [column for column in ["task", "model"] if column in repeated_summary.columns]
    numeric_columns = [
        column
        for column in repeated_summary.columns
        if column not in key_columns + ["repeat"] and pd.api.types.is_numeric_dtype(repeated_summary[column])
    ]

    aggregated = repeated_summary[key_columns].drop_duplicates().sort_values(key_columns).reset_index(drop=True)
    grouped = repeated_summary.groupby(key_columns, sort=False)

    for column in numeric_columns:
        aggregated[f"{column}_mean"] = grouped[column].mean().to_numpy()
        aggregated[f"{column}_sem"] = grouped[column].sem(ddof=1).fillna(0.0).to_numpy()

    return aggregated, repeated_summary

SHARED_TASKS = list_tasks()
CONFIG = GenerationConfig(
    temperature=0.0,
    num_predict=256,
    timeout_seconds=300,
)

print("Running shared-runner tasks:", SHARED_TASKS)
print("Models:", MODELS)
print("Output root:", OUTPUT_ROOT)
print("Sample size:", SAMPLE_SIZE)
print("Repeats per task:", NUM_REPEATS)


Running shared-runner tasks: ['gec', 'intent', 'mt_eng', 'mt_fas', 'mt_jpn', 'summarization', 'toxicity']
Models: ['qwen2.5:3b', 'gemma2:2b', 'mistral:7b', 'qwen2.5:7b-instruct']
Output root: results/colab_suite


In [11]:
task_summaries = {}
task_predictions = {}
task_repeat_summaries = {}
combined_summaries = []

for task_name in SHARED_TASKS:
    print(f"Running task: {task_name}")
    repeat_summaries = []
    repeat_predictions = []

    for repeat_index in range(NUM_REPEATS):
        repeat_number = repeat_index + 1
        repeat_seed = RANDOM_STATE + repeat_index
        print(f"  Repeat {repeat_number}/{NUM_REPEATS} with seed {repeat_seed}")
        summary, raw = run_task(
            task_name=task_name,
            models=MODELS,
            sample_size=SAMPLE_SIZE,
            random_state=repeat_seed,
            config=CONFIG,
        )
        summary = summary.copy()
        raw = raw.copy()
        summary["repeat"] = repeat_number
        raw["repeat"] = repeat_number
        repeat_output_dir = OUTPUT_ROOT / task_name / f"repeat_{repeat_number:02d}"
        save_run_outputs(summary, raw, repeat_output_dir, task_name)
        repeat_summaries.append(summary)
        repeat_predictions.append(raw)
        print(f"  Saved repeat outputs to {repeat_output_dir}")

    aggregated_summary, repeated_summary = aggregate_repeated_summaries(repeat_summaries)
    repeated_predictions = pd.concat(repeat_predictions, ignore_index=True)
    task_output_dir = OUTPUT_ROOT / task_name
    aggregated_summary.to_csv(task_output_dir / f"{task_name}_summary_with_sem.csv", index=False)
    repeated_summary.to_csv(task_output_dir / f"{task_name}_repeat_summaries.csv", index=False)
    repeated_predictions.to_csv(task_output_dir / f"{task_name}_repeat_predictions.csv", index=False)

    task_summaries[task_name] = aggregated_summary
    task_predictions[task_name] = repeated_predictions
    task_repeat_summaries[task_name] = repeated_summary
    combined_summaries.append(aggregated_summary)
    print(f"Saved aggregated outputs to {task_output_dir}")

combined_summary = pd.concat(combined_summaries, ignore_index=True)
combined_summary_path = OUTPUT_ROOT / "all_tasks_summary_with_sem.csv"
combined_summary.to_csv(combined_summary_path, index=False)
print(f"Saved combined summary to {combined_summary_path}")

combined_summary


Running task: gec
Saved benchmark outputs to results/colab_suite/gec
Running task: intent
Saved benchmark outputs to results/colab_suite/intent
Running task: mt_eng
Saved benchmark outputs to results/colab_suite/mt_eng
Running task: mt_fas
Saved benchmark outputs to results/colab_suite/mt_fas
Running task: mt_jpn
Saved benchmark outputs to results/colab_suite/mt_jpn
Running task: summarization


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/153 [00:00<?, ?B/s]

hugginface_dataset.csv:   0%|          | 0.00/289M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8395 [00:00<?, ? examples/s]

...

offenseval-gr-test-v1.tsv:   0%|          | 0.00/301k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Saved benchmark outputs to results/colab_suite/toxicity
Saved combined summary to results/colab_suite/all_tasks_summary.csv


,model,samples,exact_match,wer_vs_reference,cer_vs_reference,wer_vs_original,cer_vs_original,changed_input_rate,avg_latency_seconds,task,accuracy,macro_f1
0,qwen2.5:3b,10,0.0,11.403509,5.858855,9.734513,5.157401,0.9,9.179100,gec,NaN,NaN
1,qwen2.5:7b-instruct,10,0.2,13.157895,6.258322,11.946903,5.626256,0.9,8.171395,gec,NaN,NaN
2,gemma2:2b,10,0.1,32.456140,30.426099,30.088496,29.738781,0.8,6.456766,gec,NaN,NaN
3,mistral:7b,10,0.0,53.947368,47.470040,53.097345,47.019424,1.0,8.511833,gec,NaN,NaN
4,gemma2:2b,10,NaN,NaN,NaN,NaN,NaN,NaN,0.348796,intent,0.9,0.866667
5,qwen2.5:7b-instruct,10,NaN,NaN,NaN,NaN,NaN,NaN,0.777655,intent,0.8,0.577778
6,mistral:7b,10,NaN,NaN,NaN,NaN,NaN,NaN,0.479701,intent,0.8,0.500000
7,qwen2.5:3b,10,NaN,NaN,NaN,NaN,NaN,NaN,0.579225,intent,0.7,0.380952
8,qwen2.5:7b-instruct,10,NaN,61.299036,45.201329,NaN,NaN,NaN,1.682374,mt_eng,NaN,NaN
9,gemma2:2b,10,NaN,67.326051,44.897307,NaN,NaN,NaN,0.616465,mt_eng,NaN,NaN


In [12]:
METRIC_PREFERENCES = [
    "exact_match_mean",
    "accuracy_mean",
    "macro_f1_mean",
    "wer_vs_reference_mean",
    "cer_vs_reference_mean",
    "avg_latency_seconds_mean",
]

LOWER_IS_BETTER = {"wer_vs_reference_mean", "cer_vs_reference_mean", "avg_latency_seconds_mean"}

performance_rows = []

for task_name in SHARED_TASKS:
    summary = task_summaries[task_name].copy()
    print(f"\n=== {task_name} ===")
    display(summary)

    metric = next((column for column in METRIC_PREFERENCES if column in summary.columns), None)
    if metric is None:
        continue

    sem_column = metric.replace("_mean", "_sem")
    ascending = metric in LOWER_IS_BETTER
    ranked = summary.sort_values(metric, ascending=ascending).reset_index(drop=True)
    best_row = ranked.iloc[0]
    performance_rows.append(
        {
            "task": task_name,
            "primary_metric": metric,
            "best_model": best_row["model"],
            "best_score_mean": best_row[metric],
            "best_score_sem": best_row[sem_column] if sem_column in best_row.index else 0.0,
        }
    )

performance_by_task = pd.DataFrame(performance_rows)
print("\n=== Best Performance Per Task ===")
display(performance_by_task)

combined_summary



=== gec ===


,model,samples,exact_match,wer_vs_reference,cer_vs_reference,wer_vs_original,cer_vs_original,changed_input_rate,avg_latency_seconds,task
0,qwen2.5:3b,10,0.0,11.403509,5.858855,9.734513,5.157401,0.9,9.179100,gec
3,qwen2.5:7b-instruct,10,0.2,13.157895,6.258322,11.946903,5.626256,0.9,8.171395,gec
1,gemma2:2b,10,0.1,32.456140,30.426099,30.088496,29.738781,0.8,6.456766,gec
2,mistral:7b,10,0.0,53.947368,47.470040,53.097345,47.019424,1.0,8.511833,gec



=== intent ===


,task,model,samples,accuracy,macro_f1,avg_latency_seconds
1,intent,gemma2:2b,10,0.9,0.866667,0.348796
3,intent,qwen2.5:7b-instruct,10,0.8,0.577778,0.777655
2,intent,mistral:7b,10,0.8,0.500000,0.479701
0,intent,qwen2.5:3b,10,0.7,0.380952,0.579225



=== mt_eng ===


...


=== summarization ===


,task,model,samples,wer_vs_reference,cer_vs_reference,avg_latency_seconds
0,summarization,qwen2.5:3b,10,701.402863,484.626779,6.620048
3,summarization,qwen2.5:7b-instruct,10,720.265716,512.949091,13.876888
1,summarization,gemma2:2b,10,776.254739,581.674704,5.336306
2,summarization,mistral:7b,10,794.714922,560.726007,14.360580



=== toxicity ===


,task,model,samples,accuracy,macro_f1,avg_latency_seconds
0,toxicity,qwen2.5:3b,10,0.0,0.0,0.493892
1,toxicity,gemma2:2b,10,0.0,0.0,0.295513
2,toxicity,mistral:7b,10,0.0,0.0,0.164353
3,toxicity,qwen2.5:7b-instruct,10,0.0,0.0,0.609350


,model,samples,exact_match,wer_vs_reference,cer_vs_reference,wer_vs_original,cer_vs_original,changed_input_rate,avg_latency_seconds,task,accuracy,macro_f1
0,qwen2.5:3b,10,0.0,11.403509,5.858855,9.734513,5.157401,0.9,9.179100,gec,NaN,NaN
1,qwen2.5:7b-instruct,10,0.2,13.157895,6.258322,11.946903,5.626256,0.9,8.171395,gec,NaN,NaN
2,gemma2:2b,10,0.1,32.456140,30.426099,30.088496,29.738781,0.8,6.456766,gec,NaN,NaN
3,mistral:7b,10,0.0,53.947368,47.470040,53.097345,47.019424,1.0,8.511833,gec,NaN,NaN
4,gemma2:2b,10,NaN,NaN,NaN,NaN,NaN,NaN,0.348796,intent,0.9,0.866667
5,qwen2.5:7b-instruct,10,NaN,NaN,NaN,NaN,NaN,NaN,0.777655,intent,0.8,0.577778
6,mistral:7b,10,NaN,NaN,NaN,NaN,NaN,NaN,0.479701,intent,0.8,0.500000
7,qwen2.5:3b,10,NaN,NaN,NaN,NaN,NaN,NaN,0.579225,intent,0.7,0.380952
8,qwen2.5:7b-instruct,10,NaN,61.299036,45.201329,NaN,NaN,NaN,1.682374,mt_eng,NaN,NaN
9,gemma2:2b,10,NaN,67.326051,44.897307,NaN,NaN,NaN,0.616465,mt_eng,NaN,NaN


In [13]:
METRIC_PREFERENCES = [
    "exact_match",
    "accuracy",
    "macro_f1",
    "wer_vs_reference",
    "cer_vs_reference",
    "avg_latency_seconds",
]

LOWER_IS_BETTER = {"wer_vs_reference", "cer_vs_reference", "avg_latency_seconds"}

performance_rows = []

for task_name in SHARED_TASKS:
    summary = task_summaries[task_name].copy()
    print(f"\n=== {task_name} ===")
    display(summary)

    metric = next((column for column in METRIC_PREFERENCES if column in summary.columns), None)
    if metric is None:
        continue

    ascending = metric in LOWER_IS_BETTER
    ranked = summary.sort_values(metric, ascending=ascending).reset_index(drop=True)
    best_row = ranked.iloc[0]
    performance_rows.append(
        {
            "task": task_name,
            "primary_metric": metric,
            "best_model": best_row["model"],
            "best_score": best_row[metric],
            "samples": best_row.get("samples"),
        }
    )

performance_by_task = pd.DataFrame(performance_rows)
print("\n=== Best Performance Per Task ===")
display(performance_by_task)

combined_summary



=== gec ===


,model,samples,exact_match,wer_vs_reference,cer_vs_reference,wer_vs_original,cer_vs_original,changed_input_rate,avg_latency_seconds,task
0,qwen2.5:3b,10,0.0,11.403509,5.858855,9.734513,5.157401,0.9,9.179100,gec
3,qwen2.5:7b-instruct,10,0.2,13.157895,6.258322,11.946903,5.626256,0.9,8.171395,gec
1,gemma2:2b,10,0.1,32.456140,30.426099,30.088496,29.738781,0.8,6.456766,gec
2,mistral:7b,10,0.0,53.947368,47.470040,53.097345,47.019424,1.0,8.511833,gec



=== intent ===


,task,model,samples,accuracy,macro_f1,avg_latency_seconds
1,intent,gemma2:2b,10,0.9,0.866667,0.348796
3,intent,qwen2.5:7b-instruct,10,0.8,0.577778,0.777655
2,intent,mistral:7b,10,0.8,0.500000,0.479701
0,intent,qwen2.5:3b,10,0.7,0.380952,0.579225



=== mt_eng ===


...


=== toxicity ===


,task,model,samples,accuracy,macro_f1,avg_latency_seconds
0,toxicity,qwen2.5:3b,10,0.0,0.0,0.493892
1,toxicity,gemma2:2b,10,0.0,0.0,0.295513
2,toxicity,mistral:7b,10,0.0,0.0,0.164353
3,toxicity,qwen2.5:7b-instruct,10,0.0,0.0,0.609350



=== Best Performance Per Task ===


,task,primary_metric,best_model,best_score,samples
0,gec,exact_match,qwen2.5:7b-instruct,0.200000,10
1,intent,accuracy,gemma2:2b,0.900000,10
2,mt_eng,wer_vs_reference,qwen2.5:7b-instruct,61.299036,10
3,mt_fas,wer_vs_reference,qwen2.5:3b,108.144928,10
4,mt_jpn,wer_vs_reference,qwen2.5:7b-instruct,100.000000,10
5,summarization,wer_vs_reference,qwen2.5:3b,701.402863,10
6,toxicity,accuracy,qwen2.5:3b,0.000000,10


,model,samples,exact_match,wer_vs_reference,cer_vs_reference,wer_vs_original,cer_vs_original,changed_input_rate,avg_latency_seconds,task,accuracy,macro_f1
0,qwen2.5:3b,10,0.0,11.403509,5.858855,9.734513,5.157401,0.9,9.179100,gec,NaN,NaN
1,qwen2.5:7b-instruct,10,0.2,13.157895,6.258322,11.946903,5.626256,0.9,8.171395,gec,NaN,NaN
2,gemma2:2b,10,0.1,32.456140,30.426099,30.088496,29.738781,0.8,6.456766,gec,NaN,NaN
3,mistral:7b,10,0.0,53.947368,47.470040,53.097345,47.019424,1.0,8.511833,gec,NaN,NaN
4,gemma2:2b,10,NaN,NaN,NaN,NaN,NaN,NaN,0.348796,intent,0.9,0.866667
5,qwen2.5:7b-instruct,10,NaN,NaN,NaN,NaN,NaN,NaN,0.777655,intent,0.8,0.577778
6,mistral:7b,10,NaN,NaN,NaN,NaN,NaN,NaN,0.479701,intent,0.8,0.500000
7,qwen2.5:3b,10,NaN,NaN,NaN,NaN,NaN,NaN,0.579225,intent,0.7,0.380952
8,qwen2.5:7b-instruct,10,NaN,61.299036,45.201329,NaN,NaN,NaN,1.682374,mt_eng,NaN,NaN
9,gemma2:2b,10,NaN,67.326051,44.897307,NaN,NaN,NaN,0.616465,mt_eng,NaN,NaN
